# Web3 Python 100本ノック：第1章
## §1-4：新規ウォレットを生成し、自分だけの秘密鍵を手に入れよ

ブロックチェーンの世界に参加するための「第一歩」となるのが、自分自身のウォレット（アカウント）の作成です。
ウォレットの作成とは、本質的には**「絶対に推測されないランダムな文字列（秘密鍵）」**を数学的に生成することであり、ブロックチェーンに接続しなくても（完全にオフラインで）、行うことができます。

このノートでは、あなたがキーボードから入力した文字列を「乱数の種（エントロピー）」としてブレンドし、世界に一つだけのウォレットを対話的に生成します。


> **注意**: 各セルを順番に実行してください。セルの実行には `Shift + Enter` を押します。

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/)

## 動かす前に、キー作成の仕組みを理解しよう。
Web3に触れる際、多くの人が最初に「MetaMask（メタマスク）」などのウォレットアプリをインストールし、
12個や24個の英単語（シークレットリカバリーフレーズ）をメモしてアカウントを「登録」した経験があると思います。

しかし、実はブロックチェーンの世界には **「ウォレットのユーザー登録センター」のようなものは存在しません。**

ウォレットの正体は、単なる**「ランダムに生成された非常に巨大な数字（秘密鍵）」**と、そこから暗号理論の計算式（楕円曲線暗号）によって導き出された**「公開アドレス（0x...）」**のペアに過ぎません。
MetaMaskがやっていることは、あの12個の英単語を「乱数の種」として使い、あなたのデバイスの内部（手元）でこっそりこの数学的な計算を行っているだけなのです。

つまり、どこかのサーバーに登録の許可を得る必要はなく、インターネットから完全に切断されたオフラインの環境であっても、**計算式さえ叩けば誰でも合法的に自分だけのウォレットを生成することができます。**

### smallサンプルで理解しよう

スモールサンプルとして、以下の3ステップで順番に証明します。
- 秘密鍵の正体は、単なる「32バイト（256ビット）の数値」であること。
- どんな文字列からでも、ハッシュ関数を使えば「自分だけの数値（鍵）」が作れること。
- 宇宙規模の巨大な空間（$2^{256}$）を使うため、他者との重複（衝突）は事実上あり得ないこと。

下のセルを実行してみましょう。

In [ ]:
import hashlib
from eth_account import Account


print("=== ウォレットの裏側を暴く：暗号理論の体感スクリプト ===\n")

# -------------------------------------------------------------
# 1. 秘密鍵は「ただの数字（バイト列）」に過ぎないことの証明
# -------------------------------------------------------------
print("[1] 秘密鍵の手作り体験")
# 例として"sample_secret_2026"という簡単な文字列を、SHA-256という暗号アルゴリズムでハッシュ化（32バイトの数値に変換）します。
# ※実際の運用でこんな簡単な文字列を使うと一瞬でハッキングされますが、実験用としては十分です。
my_word = "sample_secret_2026" 

# 文字列をバイト列に変換し、SHA256でハッシュ化
raw_hash = hashlib.sha256(my_word.encode()).digest()

# ハッシュ化された32バイトのデータを、16進数の文字列（0x...）に変換
manual_private_key = "0x" + raw_hash.hex()

print(f"元データ : {my_word}")
print(f"手作り秘密鍵: {manual_private_key}\n")


# -------------------------------------------------------------
# 2. 手作りの鍵から公開アドレスを計算（登録不要の証明）
# -------------------------------------------------------------
print("[2] 楕円曲線暗号によるアドレスの導出")
print("この手作りの秘密鍵を eth_account に渡し、数学的な計算だけでアドレスを導き出します...")

# 外部のサーバーには一切通信せず、ローカルのCPU計算だけでアドレスが確定します
account = Account.from_key(manual_private_key)

print(f"導出されたアドレス: {account.address}")
print(" この時点で、このアドレスは世界中であなただけが操作できる口座として『すでに存在』しています。\n")


# -------------------------------------------------------------
# 3. なぜ誰とも重複しないのか？（圧倒的な数値スケールの体感）
# -------------------------------------------------------------
print("[3] 『登録』が不要な理由：宇宙規模の確率空間")

# 秘密鍵（32バイト）が表現できるパターンの総数は「2の256乗」です。
total_patterns = 2 ** 256

# 比較対象：地球上の砂粒の数（約 10の19乗 と言われています）
sand_grains = 10 ** 19

print("秘密鍵として作れるパターンの総数:")
print(f"{total_patterns}")

print(f"\nこれは、地球上のすべての砂粒の数の 約 {total_patterns // sand_grains} 倍です。")
print(" 誰も管理サーバーを持っていなくても、『偶然他の人と同じ鍵を作ってしまう確率』が実質ゼロであるため、システムが成立するのです。")


=== ウォレットの裏側を暴く：暗号理論の体感スクリプト ===

[1] 秘密鍵の手作り体験
元データ : lumenHero_secret_2026
手作り秘密鍵: 0x8261d4aa5274deb7f11c047a1ccc45de3a469d31d2e3e36e2a3484b823a09e86

[2] 楕円曲線暗号によるアドレスの導出
この手作りの秘密鍵を eth_account に渡し、数学的な計算だけでアドレスを導き出します...
導出されたアドレス: 0xDEFec1B904Fa8ac57d68B0A93316818bA4062A9c
 この時点で、このアドレスは世界中であなただけが操作できる口座として『すでに存在』しています。

[3] 『登録』が不要な理由：宇宙規模の確率空間
秘密鍵として作れるパターンの総数:
115792089237316195423570985008687907853269984665640564039457584007913129639936

これは、地球上のすべての砂粒の数の 約 11579208923731619542357098500868790785326998466564056403945 倍です。
 誰も管理サーバーを持っていなくても、『偶然他の人と同じ鍵を作ってしまう確率』が実質ゼロであるため、システムが成立するのです。


実行してみると、わかる通り、文字列→秘密鍵をローカルで作成できることが確認できます。　

またこの秘密鍵をもとに楕円曲線暗号を用いてアドレス変換し新規addressを作成できました。

砂粒の比較の通り、現実的にアドレスが完全一致することはまずありえないため、中央集権サーバなどで管理しなくてもこのアルゴリズムを利用して作成すればまず同じaddressが出来上がることはありません。

## 実践：対話型ウォレットジェネレーター

理論がわかったところで、実際にPythonを使ってオフラインでウォレットを生成してみましょう。

このノートでは、あなたがキーボードから入力した文字列を「乱数の種（エントロピー）」としてブレンドし、世界に一つだけのウォレットを対話的に生成します。

> **注意**: 各セルを順番に実行してください。セルの実行には `Shift + Enter` を押します。

### my_word(キー生成の種)の入力

In [1]:
from eth_account import Account
import secrets

print("ウォレット作成。\n")

# 生成の種となる文字列入力を受け付ける
user_entropy = input(" あなたのオリジナル文字列を入力: ")

ウォレット作成。



In [3]:
print(f"入力したオリジナル文字列の確認\n{user_entropy}\n")

入力したオリジナル文字列の確認
secretmessage



### キー生成

In [ ]:
print("\n 鍵を生成中...")

# ユーザーの入力文字列と、システムが生成した強力な乱数を結合して「乱数の種」を作る
combined_entropy = user_entropy + secrets.token_hex(32)

# eth_accountを使ってウォレットを新規生成
new_account = Account.create(combined_entropy)

print("\n あなた専用の新しいウォレットが完成しました！")
print("=" * 60)
print(f" 公開アドレス (Public Address) : {new_account.address}")
print(f" 秘密鍵 (Private Key)        : {new_account.key.hex()}")
print("=" * 60)
print("\n この秘密鍵は他人に教えないでください。")
print("〇『公開アドレス』は銀行の口座番号のようなもので、他人に教えても問題ありません。")
print("〇 しかし、『秘密鍵』は銀行の暗証番号そのものです。絶対に他人に教えたり、ネット上に公開しないことが重要です。")
print(" ※今回のスクリプトは学習用のサンプルなので、問題はないですが、実際の運用では絶対に秘密鍵を公開しないでください。")


 鍵を生成中...

 あなた専用の新しいウォレットが完成しました！
 公開アドレス (Public Address) : 0xEF71DE30694fC23C795E09f0A59B83dbba243b42
 秘密鍵 (Private Key)        : e1328364b817ad2adff416f3d9ab5b2c39681ab0d59cbab135d4925b9c2de475

 この秘密鍵は他人に教えないでください。
『公開アドレス』は銀行の口座番号のようなもので、他人に教えても問題ありません。
 しかし、『秘密鍵』は銀行の暗証番号そのものです。絶対に他人に教えたり、ネット上に公開しないことが重要です。
 ※今回のスクリプトは学習用のサンプルなので、問題はないですが、実際の運用では絶対に秘密鍵を公開しないでください。
